<a href="https://colab.research.google.com/github/akmlrhim/covid19-analysis-data/blob/main/index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q plotly pyarrow streamlit scipy

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.signal import find_peaks

for d in ["data/raw", "data/processed"]:
    os.makedirs(d, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 65.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
urls = [
    "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv",
    "https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv",
]

raw = None
for u in urls:
    try:
        raw = pd.read_csv(u, parse_dates=["date"])
        print("Berhasil dari:", u)
        break
    except Exception as e:
        print("Gagal:", u, "->", type(e).__name__)

if raw is None:
    raise RuntimeError("Semua URL gagal, pakai Cara 2 (unggah manual).")

raw.to_csv("data/raw/owid-covid-data.csv", index=False)
print(raw.shape)
print(raw["date"].min(), "->", raw["date"].max())
raw.head()

Berhasil dari: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
(429435, 67)
2020-01-01 00:00:00 -> 2024-08-14 00:00:00


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [5]:
# 1) Samakan nama kolom
raw = raw.rename(columns={"country": "location", "entity": "location"})
print("Kolom tersedia:", raw.columns.tolist())

asean = ["Indonesia", "Malaysia", "Singapore", "Thailand", "Vietnam", "Philippines"]
df = raw[raw["location"].isin(asean)].copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["location", "date"]).reset_index(drop=True)

# 2) Kolom yang tidak ada dibuat/diturunkan
if "new_cases" not in df.columns:
    df["new_cases"] = df.groupby("location")["total_cases"].diff()
if "new_deaths" not in df.columns:
    df["new_deaths"] = df.groupby("location")["total_deaths"].diff()
if "people_fully_vaccinated_per_hundred" not in df.columns:
    df["people_fully_vaccinated_per_hundred"] = np.nan
    print("PERINGATAN: data vaksinasi tidak ada, Sel 7 akan dilewati.")

# 3) Populasi: pakai kolom bawaan, kalau tidak ada pakai nilai perkiraan (sekitar 2021)
if "population" not in df.columns:
    pop = {"Indonesia": 273_500_000, "Malaysia": 33_600_000, "Singapore": 5_450_000,
           "Thailand": 71_600_000, "Vietnam": 97_500_000, "Philippines": 111_000_000}
    df["population"] = df["location"].map(pop)
    print("Populasi memakai nilai perkiraan, catat di README.")

# 4) Batasi periode
df = df[(df["date"] >= "2020-03-01") & (df["date"] <= "2022-12-31")].copy()

# 5) Nilai negatif = koreksi data
print("Nilai negatif:\n", (df[["new_cases", "new_deaths"]] < 0).sum())
df[["new_cases", "new_deaths"]] = df[["new_cases", "new_deaths"]].clip(lower=0)

# 6) Isi kolom kumulatif ke depan, vaksinasi awal = 0 jika ada datanya
for c in ["total_cases", "total_deaths", "people_fully_vaccinated_per_hundred"]:
    df[c] = df.groupby("location")[c].ffill()
if df["people_fully_vaccinated_per_hundred"].notna().any():
    df["people_fully_vaccinated_per_hundred"] = df["people_fully_vaccinated_per_hundred"].fillna(0)

# 7) Cek kualitas data
print((df.isna().mean() * 100).round(2))
print(df.groupby("location")["date"].agg(["min", "max", "count"]))

Kolom tersedia: ['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients', 'icu_patients_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests', 'total_tests_per_thousand', 'new_tests_per_thousand', 'new_tests_smoothed', 'new_tests_smoothed_per_thousand', 'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_boosters', 'new_vaccinations', 'new_vaccinations_smoothed', 'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_h

In [6]:
df["cases_per_100k"]  = df["new_cases"]  / df["population"] * 100_000
df["deaths_per_100k"] = df["new_deaths"] / df["population"] * 100_000

for src, dst in [("cases_per_100k", "cases_ma7"), ("deaths_per_100k", "deaths_ma7")]:
    df[dst] = df.groupby("location")[src].transform(lambda s: s.rolling(7, min_periods=1).mean())

df["cfr"] = (df["total_deaths"] / df["total_cases"] * 100).replace([np.inf, -np.inf], np.nan)

df.to_parquet("data/processed/covid_asean.parquet", index=False)
df.tail()

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million,cases_per_100k,deaths_per_100k,cases_ma7,deaths_ma7,cfr
9468,VNM,Asia,Vietnam,2022-12-27,11524202.0,0.0,207.43,43184.0,0.0,0.57,...,98186856,NaN,NaN,NaN,NaN,0.0,0.0,0.211259,0.000582,0.374724
9469,VNM,Asia,Vietnam,2022-12-28,11524202.0,0.0,207.43,43184.0,0.0,0.57,...,98186856,NaN,NaN,NaN,NaN,0.0,0.0,0.211259,0.000582,0.374724
9470,VNM,Asia,Vietnam,2022-12-29,11524202.0,0.0,207.43,43184.0,0.0,0.57,...,98186856,NaN,NaN,NaN,NaN,0.0,0.0,0.211259,0.000582,0.374724
9471,VNM,Asia,Vietnam,2022-12-30,11524202.0,0.0,207.43,43184.0,0.0,0.57,...,98186856,NaN,NaN,NaN,NaN,0.0,0.0,0.211259,0.000582,0.374724
9472,VNM,Asia,Vietnam,2022-12-31,11524202.0,0.0,207.43,43184.0,0.0,0.57,...,98186856,NaN,NaN,NaN,NaN,0.0,0.0,0.211259,0.000582,0.374724


In [7]:
ringkasan = df.groupby("location").agg(
    total_kasus=("new_cases", "sum"),
    total_kematian=("new_deaths", "sum"),
    populasi=("population", "max"),
).assign(
    kasus_per_100k=lambda x: x.total_kasus / x.populasi * 100_000,
    kematian_per_100k=lambda x: x.total_kematian / x.populasi * 100_000,
).round(1).sort_values("kematian_per_100k", ascending=False)
ringkasan

,total_kasus,total_kematian,populasi,kasus_per_100k,kematian_per_100k
location,,,,,
Malaysia,5023284.0,36835.0,33938216,14801.3,108.5
Indonesia,6716124.0,160537.0,275501344,2437.8,58.3
Philippines,4059366.0,65261.0,115559008,3512.8,56.5
Thailand,4721769.0,33654.0,71697024,6585.7,46.9
Vietnam,11524186.0,43184.0,98186856,11737.0,44.0
Singapore,2193801.0,1711.0,5637022,38917.7,30.4


In [8]:
fig = px.line(df, x="date", y="cases_ma7", color="location",
              title="Kasus Harian per 100.000 Penduduk (Rata-rata 7 Hari)",
              labels={"cases_ma7": "Kasus / 100k", "date": "Tanggal", "location": "Negara"})
fig.show()

In [9]:
# Heatmap kasus per bulan x negara
bulanan = (df.assign(bulan=df["date"].dt.to_period("M").astype(str))
             .groupby(["location", "bulan"])["cases_per_100k"].sum().reset_index())
pivot = bulanan.pivot(index="location", columns="bulan", values="cases_per_100k")
px.imshow(pivot, aspect="auto", color_continuous_scale="Reds",
          title="Kasus per 100k per Bulan").show()

In [10]:
hasil = []
for negara, g in df.groupby("location"):
    g = g.reset_index(drop=True)
    idx, _ = find_peaks(g["cases_ma7"], prominence=g["cases_ma7"].max() * 0.15, distance=60)
    for i in idx:
        hasil.append({"negara": negara, "tanggal_puncak": g.loc[i, "date"],
                      "kasus_per_100k": round(g.loc[i, "cases_ma7"], 2)})

puncak = pd.DataFrame(hasil).sort_values(["negara", "tanggal_puncak"])
puncak

,negara,tanggal_puncak,kasus_per_100k
0,Indonesia,2021-02-03,4.62
1,Indonesia,2021-07-21,18.16
2,Indonesia,2022-02-23,20.21
3,Malaysia,2021-09-01,63.56
4,Malaysia,2022-03-09,85.44
5,Philippines,2021-04-07,9.04
6,Philippines,2021-09-15,18.10
7,Philippines,2022-01-19,29.66
8,Singapore,2021-11-03,65.21
9,Singapore,2022-03-02,325.00


In [11]:
negara_pilih = "Indonesia"   # ganti sesuai kebutuhan
g = df[df["location"] == negara_pilih]
p = puncak[puncak["negara"] == negara_pilih]

fig = go.Figure()
fig.add_trace(go.Scatter(x=g["date"], y=g["cases_ma7"], name="Kasus (MA7)"))
fig.add_trace(go.Scatter(x=p["tanggal_puncak"], y=p["kasus_per_100k"], mode="markers",
                         marker=dict(size=10, color="red"), name="Puncak"))
fig.update_layout(title=f"Gelombang COVID-19 {negara_pilih}",
                  xaxis_title="Tanggal", yaxis_title="Kasus / 100k")
fig.show()

In [12]:
if df["people_fully_vaccinated_per_hundred"].notna().any():
    df["tingkat_vaksin"] = pd.cut(df["people_fully_vaccinated_per_hundred"],
                                  bins=[-0.1, 5, 20, 40, 100],
                                  labels=["0-5%", "5-20%", "20-40%", ">40%"])
    vak = (df.groupby(["location", "tingkat_vaksin"], observed=True)["deaths_ma7"]
             .mean().reset_index())
    px.bar(vak, x="tingkat_vaksin", y="deaths_ma7", color="location", barmode="group",
           title="Rata-rata Kematian Harian per 100k menurut Cakupan Vaksinasi",
           labels={"deaths_ma7": "Kematian / 100k", "tingkat_vaksin": "Cakupan vaksin lengkap"}).show()
else:
    print("Data vaksinasi tidak tersedia, bagian ini dilewati.")

In [14]:
%%writefile app.py
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Dashboard COVID-19 ASEAN", layout="wide")

@st.cache_data
def load_data():
    return pd.read_parquet("data/processed/covid_asean.parquet")

df = load_data()

st.title("Dashboard Analisis COVID-19 ASEAN")
st.caption("Sumber data: Our World in Data. Periode 2020-2022.")

# ---------- Sidebar ----------
st.sidebar.header("Filter")
negara = st.sidebar.multiselect(
    "Pilih negara", sorted(df["location"].unique()),
    default=[n for n in ["Indonesia", "Malaysia"] if n in df["location"].unique()],
)
tgl = st.sidebar.date_input("Rentang tanggal", [df["date"].min(), df["date"].max()])

label_metrik = {
    "Kasus per 100k (MA7)": "cases_ma7",
    "Kematian per 100k (MA7)": "deaths_ma7",
    "CFR kumulatif (%)": "cfr",
}
if df["people_fully_vaccinated_per_hundred"].notna().any():
    label_metrik["Vaksinasi lengkap (per 100 penduduk)"] = "people_fully_vaccinated_per_hundred"

pilihan = st.sidebar.selectbox("Metrik", list(label_metrik))
metrik = label_metrik[pilihan]

if not negara or len(tgl) != 2:
    st.info("Pilih minimal satu negara dan rentang tanggal lengkap.")
    st.stop()

d = df[df["location"].isin(negara) &
       df["date"].between(pd.to_datetime(tgl[0]), pd.to_datetime(tgl[1]))]

# ---------- KPI ----------
c1, c2, c3 = st.columns(3)
c1.metric("Total Kasus", f"{d['new_cases'].sum():,.0f}")
c2.metric("Total Kematian", f"{d['new_deaths'].sum():,.0f}")
c3.metric("Negara Dipilih", len(negara))

# ---------- Tab ----------
tab1, tab2, tab3 = st.tabs(["Tren Waktu", "Perbandingan", "Catatan & Keterbatasan"])

with tab1:
    st.plotly_chart(
        px.line(d, x="date", y=metrik, color="location",
                labels={metrik: pilihan, "date": "Tanggal", "location": "Negara"}),
        use_container_width=True,
    )

with tab2:
    agg = d.groupby("location").agg(
        kasus_per_100k=("cases_per_100k", "sum"),
        kematian_per_100k=("deaths_per_100k", "sum"),
    ).reset_index()
    a, b = st.columns(2)
    a.plotly_chart(px.bar(agg, x="location", y="kasus_per_100k", color="location",
                          title="Total Kasus per 100k"), use_container_width=True)
    b.plotly_chart(px.bar(agg, x="location", y="kematian_per_100k", color="location",
                          title="Total Kematian per 100k"), use_container_width=True)

with tab3:
    st.markdown("""
    - Angka dinormalisasi per 100.000 penduduk agar adil antar negara.
    - Rata-rata bergerak 7 hari dipakai untuk meredam fluktuasi harian.
    - Perbedaan tingkat testing dan sistem pelaporan membuat perbandingan kasus tidak sepenuhnya setara.
    - Hubungan vaksinasi dan kematian bersifat korelasi, bukan sebab-akibat.
    - Nilai harian negatif (koreksi data) diubah menjadi 0.
    """)

Overwriting app.py


In [15]:
%%writefile requirements.txt
streamlit
pandas
numpy
plotly
pyarrow

Writing requirements.txt


In [16]:
!zip -r covid-analysis.zip app.py requirements.txt data/processed

from google.colab import files
files.download("covid-analysis.zip")

  adding: app.py (deflated 55%)
  adding: requirements.txt (stored 0%)
  adding: data/processed/ (stored 0%)
  adding: data/processed/covid_asean.parquet (deflated 30%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>